# Lesson 1: Energy Data Management

## Single Neuron Perceptron with Synthetic Energy Data

This notebook presents a structured introductory classification example for the course module on artificial intelligence in smart cities and energy management.

The lesson implements a **single-neuron perceptron** trained on synthetic energy-related data and evaluates the model as a binary classifier.

Classification target:

- Class `1`: high energy demand
- Class `0`: not high energy demand


## Step 1: Import the Required Tools

This step imports the Python modules required for data generation, model implementation, and evaluation.

- `random` is used to generate synthetic data and initialize the perceptron weights.
- `dataclass` helps us define a simple structure for each training sample.
- `sklearn.metrics` gives us standard evaluation measures such as accuracy, precision, recall, F1-score, and the confusion matrix.

In [13]:
import random
from dataclasses import dataclass

from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, f1_score, precision_score, recall_score

## Step 2: Create a `Sample` Class

This class represents **one data point** in our dataset.

Each sample contains:

- `features`: the input values used by the perceptron
- `label`: the expected output (`0` or `1`)

This makes the dataset easier to read and organize.

In [14]:
@dataclass
class Sample:
    # Step 2.1: Store the input features for one observation.
    features: list[float]
    
    # Step 2.2: Store the correct class label for that observation.
    label: int

## Step 3: Build the `Perceptron` Class

This class implements a **single artificial neuron**.

The perceptron will:

- store weights and a bias
- calculate a weighted sum
- convert that sum into a class prediction
- update its parameters during training

This is the core model in our example.

In [15]:
class Perceptron:
    """
    Step 3 class:
    A single-neuron perceptron for binary classification.
    """

    def __init__(self, n_features: int, learning_rate: float = 0.01) -> None:
        """
        Step 3.1:
        Initialize the perceptron.

        Parameters:
        - n_features: number of input features
        - learning_rate: controls how much weights change during learning
        """
        self.learning_rate = learning_rate
        
        # Initialize weights with small random values.
        self.weights = [random.uniform(-0.5, 0.5) for _ in range(n_features)]
        
        # Initialize the bias with a small random value.
        self.bias = random.uniform(-0.5, 0.5)

    def predict_raw(self, features: list[float]) -> float:
        """
        Step 3.2:
        Compute the raw weighted sum before classification.

        This is: (w1*x1 + w2*x2 + ... + wn*xn) + bias
        """
        total = self.bias
        for weight, feature in zip(self.weights, features):
            total += weight * feature
        return total

    def predict(self, features: list[float]) -> int:
        """
        Step 3.3:
        Convert the raw output into a class label.

        Rule:
        - if raw output >= 0, predict class 1
        - otherwise, predict class 0
        """
        return 1 if self.predict_raw(features) >= 0 else 0

    def train(self, dataset: list[Sample], epochs: int = 25) -> None:
        """
        Step 3.4:
        Train the perceptron using the perceptron learning rule.

        For each sample:
        1. predict the output
        2. compare it with the true label
        3. update weights and bias if there is an error
        """
        for epoch in range(epochs):
            errors = 0
            random.shuffle(dataset)

            for sample in dataset:
                prediction = self.predict(sample.features)
                update = self.learning_rate * (sample.label - prediction)

                if update != 0:
                    errors += 1

                # Update each weight using the corresponding feature value.
                for index in range(len(self.weights)):
                    self.weights[index] += update * sample.features[index]

                # Update the bias term.
                self.bias += update

            print(f"Epoch {epoch + 1:02d} | misclassifications: {errors}")

## Step 4: Define Helper Functions for Feature Preparation

Before training the model, we need to convert raw values into normalized values.

Why do we normalize?

- It keeps feature values on similar scales.
- It helps the perceptron learn more smoothly.
- It makes the weighted contributions easier to interpret.

In [16]:
def normalize_temperature(temp_c: float) -> float:
    """
    Step 4.1:
    Normalize temperature to a smaller range.
    """
    return (temp_c - 10.0) / 25.0


def normalize_occupancy(occupancy: int) -> float:
    """
    Step 4.2:
    Convert occupancy into a value between 0 and 1.
    """
    return occupancy / 100.0

## Step 5: Generate One Synthetic Energy Sample

This function creates one artificial observation.

The synthetic input variables are:

- outdoor temperature
- occupancy level
- hour of day
- whether the day is a weekday

We then derive:

- whether the hour falls inside business hours
- a synthetic energy score
- the final label (`high demand` or `not high demand`)

This is not real energy data. It is a simplified educational approximation.

In [17]:
def generate_sample() -> Sample:
    """
    Step 5:
    Create one synthetic sample that mimics building energy demand conditions.
    """
    # Step 5.1: Create randomized raw values.
    temperature = random.uniform(8.0, 38.0)
    occupancy = random.randint(5, 100)
    hour = random.randint(0, 23)
    is_weekday = random.randint(0, 1)

    # Step 5.2: Derive a simple business-hours indicator.
    business_hours = 1 if 8 <= hour <= 18 else 0

    # Step 5.3: Build a synthetic energy score.
    # Hotter weather, higher occupancy, business hours, and weekdays
    # all tend to increase expected demand.
    energy_score = (
        0.9 * normalize_temperature(temperature)
        + 1.2 * normalize_occupancy(occupancy)
        + 0.7 * business_hours
        + 0.4 * is_weekday
        + random.uniform(-0.25, 0.25)
    )

    # Step 5.4: Convert the score into a binary class label.
    high_demand = 1 if energy_score > 1.55 else 0

    # Step 5.5: Build the perceptron input vector.
    features = [
        normalize_temperature(temperature),
        normalize_occupancy(occupancy),
        float(business_hours),
        float(is_weekday),
    ]

    return Sample(features=features, label=high_demand)

## Step 6: Generate a Full Dataset

A machine learning model needs many samples, not just one.

This function repeats the previous process multiple times and returns a dataset.

In [18]:
def generate_dataset(size: int) -> list[Sample]:
    """
    Step 6:
    Create a dataset with the requested number of synthetic samples.
    """
    return [generate_sample() for _ in range(size)]

## Step 7: Prepare Evaluation Functions

After training, we need a more complete evaluation.

In this step, we build helper functions that let us:

- extract the true labels and predicted labels
- compute accuracy
- compute precision, recall, and F1-score
- generate a confusion matrix
- print a classification report

In [19]:
def get_true_and_predicted_labels(model: Perceptron, dataset: list[Sample]) -> tuple[list[int], list[int]]:
    """
    Step 7:
    Extract true labels and predicted labels from a dataset.
    """
    y_true = []
    y_pred = []

    for sample in dataset:
        y_true.append(sample.label)
        y_pred.append(model.predict(sample.features))

    return y_true, y_pred


def evaluate_model(model: Perceptron, dataset: list[Sample]) -> dict:
    """
    Step 7.1:
    Compute a complete set of evaluation metrics for binary classification.
    """
    y_true, y_pred = get_true_and_predicted_labels(model, dataset)

    return {
        "accuracy": accuracy_score(y_true, y_pred),
        "precision": precision_score(y_true, y_pred, zero_division=0),
        "recall": recall_score(y_true, y_pred, zero_division=0),
        "f1": f1_score(y_true, y_pred, zero_division=0),
        "confusion_matrix": confusion_matrix(y_true, y_pred),
        "classification_report": classification_report(y_true, y_pred, zero_division=0),
        "y_true": y_true,
        "y_pred": y_pred,
    }

## Step 8: Display a Few Samples in a Readable Format

This helper function is only for presentation.

It makes the output easier for students to read.

In [20]:
def pretty_feature_row(features: list[float]) -> str:
    """
    Step 8:
    Convert one feature vector into a readable text line.
    """
    return (
        f"temp={features[0]:.2f}, "
        f"occupancy={features[1]:.2f}, "
        f"business_hours={features[2]:.0f}, "
        f"weekday={features[3]:.0f}"
    )

## Step 9: Create the Training and Test Data

We now create two datasets:

- a training set used to learn
- a test set used to evaluate generalization

We also fix the random seed so students can reproduce the same output.

In [21]:
# Step 9.1: Fix the random seed for reproducible results.
random.seed(42)

# Step 9.2: Generate the datasets.
train_data = generate_dataset(200)
test_data = generate_dataset(60)

print(f"Training samples: {len(train_data)}")
print(f"Test samples: {len(test_data)}")

Training samples: 200
Test samples: 60


## Step 10: Create and Train the Perceptron

Now we instantiate the model and train it.

Observe the number of misclassifications per epoch. As training progresses, the model should generally improve.

In [22]:
# Step 10.1: Create the perceptron.
model = Perceptron(n_features=4, learning_rate=0.05)

# Step 10.2: Train the model.
model.train(train_data, epochs=20)

Epoch 01 | misclassifications: 52
Epoch 02 | misclassifications: 32
Epoch 03 | misclassifications: 33
Epoch 04 | misclassifications: 24
Epoch 05 | misclassifications: 33
Epoch 06 | misclassifications: 21
Epoch 07 | misclassifications: 26
Epoch 08 | misclassifications: 23
Epoch 09 | misclassifications: 22
Epoch 10 | misclassifications: 18
Epoch 11 | misclassifications: 21
Epoch 12 | misclassifications: 20
Epoch 13 | misclassifications: 27
Epoch 14 | misclassifications: 20
Epoch 15 | misclassifications: 19
Epoch 16 | misclassifications: 23
Epoch 17 | misclassifications: 16
Epoch 18 | misclassifications: 17
Epoch 19 | misclassifications: 21
Epoch 20 | misclassifications: 23


## Step 11: Evaluate the Model

We now calculate:

- training accuracy
- test accuracy
- precision
- recall
- F1-score
- the confusion matrix
- the classification report

This helps us understand how well the perceptron learned the synthetic pattern.

In [23]:
# Step 11.1: Compute complete evaluation summaries.
train_results = evaluate_model(model, train_data)
test_results = evaluate_model(model, test_data)

# Step 11.2: Print the learned parameters.
print("Learned parameters")
print(f"Weights: {[round(weight, 3) for weight in model.weights]}")
print(f"Bias: {model.bias:.3f}")

# Step 11.3: Print the main evaluation scores.
print("\nTraining scores")
print(f"Accuracy:  {train_results['accuracy']:.2%}")
print(f"Precision: {train_results['precision']:.2%}")
print(f"Recall:    {train_results['recall']:.2%}")
print(f"F1-score:  {train_results['f1']:.2%}")

print("\nTest scores")
print(f"Accuracy:  {test_results['accuracy']:.2%}")
print(f"Precision: {test_results['precision']:.2%}")
print(f"Recall:    {test_results['recall']:.2%}")
print(f"F1-score:  {test_results['f1']:.2%}")

Learned parameters
Weights: [0.393, 0.422, 0.201, 0.142]
Bias: -0.617

Training scores
Accuracy:  90.50%
Precision: 98.98%
Recall:    84.35%
F1-score:  91.08%

Test scores
Accuracy:  86.67%
Precision: 96.67%
Recall:    80.56%
F1-score:  87.88%


## Step 12: Print the Confusion Matrix and Classification Report

The confusion matrix gives a detailed view of prediction quality.

For binary classification:

- top-left: true negatives
- top-right: false positives
- bottom-left: false negatives
- bottom-right: true positives

The classification report summarizes precision, recall, and F1-score for each class.

In [24]:
# Step 12.1: Print the confusion matrix for the test set.
print("Test confusion matrix")
print(test_results["confusion_matrix"])

# Step 12.2: Print the classification report for the test set.
print("\nTest classification report")
print(test_results["classification_report"])

Test confusion matrix
[[23  1]
 [ 7 29]]

Test classification report
              precision    recall  f1-score   support

           0       0.77      0.96      0.85        24
           1       0.97      0.81      0.88        36

    accuracy                           0.87        60
   macro avg       0.87      0.88      0.87        60
weighted avg       0.89      0.87      0.87        60



## Step 13: Inspect a Few Predictions

Finally, we compare predicted labels with actual labels for a few test samples.

This makes the model behavior more concrete for students.

In [25]:
# Step 13: Show a few example predictions.
for sample in test_data[:5]:
    prediction = model.predict(sample.features)
    print(
        f"{pretty_feature_row(sample.features)}"
        f" -> predicted={prediction}, actual={sample.label}"
    )

temp=0.27, occupancy=0.52, business_hours=1, weekday=0 -> predicted=0, actual=0
temp=0.41, occupancy=0.92, business_hours=0, weekday=0 -> predicted=0, actual=0
temp=0.65, occupancy=0.53, business_hours=0, weekday=0 -> predicted=0, actual=0
temp=0.62, occupancy=0.34, business_hours=1, weekday=1 -> predicted=1, actual=1
temp=0.72, occupancy=0.64, business_hours=1, weekday=0 -> predicted=1, actual=1
